# RescueVision v2 - change detection (256px crops, long train)

In [ ]:
import subprocess, sys, os, json, time, traceback
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = Path("/tmp/RescueVision")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/AliOmarAbdelhady/RescueVision.git", str(REPO)], check=True)
sys.path.insert(0, str(REPO / "src"))

def mark(stage, ok=True, extra=""):
    """Progress ledger in /kaggle/working: survives kills where the log does not."""
    rec = {"t": time.strftime("%H:%M:%S"), "stage": stage, "ok": bool(ok), "extra": str(extra)[:1500]}
    with open(WORK / "_progress.txt", "a") as f:
        f.write(json.dumps(rec) + "\n")
    (WORK / "_status.json").write_text(json.dumps(rec, indent=1))
    print("[mark]", rec, flush=True)

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
mark("setup:start")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "albumentations==2.0.8"], check=False)
import albumentations as _A
mark("setup:albumentations", extra=_A.__version__)
print("albumentations", _A.__version__)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch"], check=False)
import segmentation_models_pytorch as smp
print("smp", smp.__version__)
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NW = 0
PIN = False
print("torch", torch.__version__, "| device:", DEVICE)
print(subprocess.run(["df", "-h", "/tmp", "/kaggle/working"], capture_output=True, text=True).stdout)
mark("setup:done", extra=str(DEVICE))

import rescuevision.training.logging_utils as _lu
_orig_log = _lu.CSVLogger.log
def _marked_log(self, metrics, epoch):
    try:
        mark(f"epoch{epoch}", extra=str({k: round(float(v), 4) for k, v in metrics.items()
                                         if isinstance(v, (int, float))})[:400])
    except Exception:
        pass
    return _orig_log(self, metrics, epoch)
_lu.CSVLogger.log = _marked_log
print("epoch heartbeat patch applied")

In [ ]:
import numpy as np
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2

from rescuevision.constants import IMAGENET_MEAN, IMAGENET_STD
from rescuevision.data import datasets as _D
from PIL import Image as _PIL
from pathlib import Path as _PPath

def _load_arr(path):
    path = _PPath(path)
    if path.suffix == ".npy":
        return np.load(path)
    return np.array(_PIL.open(path))

def _find_tile(directory, tile_id):
    for ext in (".png", ".jpg", ".jpeg", ".npy"):
        p = _PPath(directory) / f"{tile_id}{ext}"
        if p.exists():
            return p
    raise FileNotFoundError(f"{tile_id} in {directory}")

def _tile_dataset_init(self, processed_dir, split_csv, split="train", mask_type="building_binary",
                       transform=None, image_size=512, debug_limit=None):
    import pandas as pd
    self.processed_dir = _PPath(processed_dir)
    self.images_dir = self.processed_dir / "images" / "pre"
    self.pre_dir = self.processed_dir / "images" / "pre"
    self.post_dir = self.processed_dir / "images" / "post"
    if type(self).__name__ == "XBDDamageSegmentationDataset":
        self.masks_dir = self.processed_dir / "masks" / "damage_multiclass"
    elif type(self).__name__ == "XBDChangeDetectionDataset":
        self.masks_dir = self.processed_dir / "masks" / "change_binary"
    else:
        self.masks_dir = self.processed_dir / "masks" / mask_type
    self.transform = transform
    self.image_size = image_size
    splits_df = pd.read_csv(split_csv)
    tile_ids = splits_df[splits_df["split"] == split]["tile_id"].tolist()
    if debug_limit:
        tile_ids = tile_ids[:debug_limit]
    self.tile_ids = [t for t in tile_ids
                     if _find_tile(self.pre_dir, t).exists()
                     and _find_tile(self.post_dir, t).exists()
                     and _find_tile(self.masks_dir, t).exists()]
    print(f"{type(self).__name__}: split={split}, {len(self.tile_ids)} tiles")

def _xbd_seg_getitem(self, idx):
    tile_id = self.tile_ids[idx]
    image = _load_arr(_find_tile(self.pre_dir, tile_id))
    mask = _load_arr(_find_tile(self.masks_dir, tile_id))
    if self.transform:
        aug = self.transform(image=image, mask=mask)
        image, mask = aug["image"], aug["mask"]
    mask = mask.long() if torch.is_tensor(mask) else torch.from_numpy(mask).long()
    return {"image": image, "mask": mask, "tile_id": tile_id}

_D.XBDSegmentationDataset.__init__ = _tile_dataset_init
_D.XBDSegmentationDataset.__getitem__ = _xbd_seg_getitem
mark("patch:done")
print("png-aware dataset patch applied")

In [ ]:
import time
import pandas as pd
mark("tar:wait")
TAR = None
if (Path("/tmp/processed_xbd") / "metadata" / "splits_event_aware.csv").exists():
    print("processed tree already at /tmp/processed_xbd (smoke/debug path)")
else:
    for _attempt in range(40):
        for _p in Path("/kaggle/input").rglob("processed_xbd*.tar"):
            TAR = _p
            break
        if TAR is not None:
            break
        _ents = [p.name for p in Path("/kaggle/input").iterdir()] if Path("/kaggle/input").exists() else []
        print(f"[{_attempt+1}/40] waiting for tar mount, /kaggle/input: {_ents}", flush=True)
        time.sleep(30)
    assert TAR is not None, "processed_xbd_native.tar never mounted - kernel-source attach failed"
    _gib = TAR.stat().st_size / 1e9
    print("tar:", TAR, f"{_gib:.2f} GB")
    mark("tar:mounted", extra=f"{_gib:.2f}GB")
    subprocess.run(["tar", "-xf", str(TAR), "-C", "/tmp"], check=True)
PROCESSED = Path("/tmp/processed_xbd")
assert (PROCESSED / "metadata" / "splits_event_aware.csv").exists(), "extracted tree missing splits csv"
_splits = pd.read_csv(PROCESSED / "metadata" / "splits_event_aware.csv")
print("tiles per split:", _splits["split"].value_counts().to_dict())
_tiles_meta = pd.read_csv(PROCESSED / "metadata" / "tiles.csv")
print("shift columns:", [c for c in _tiles_meta.columns if "shift" in c])
mark("tar:extracted", extra=str(_splits["split"].value_counts().to_dict()))

In [ ]:
import logging, time
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
from rescuevision.training.train_change_detection import train_change_detection
from rescuevision.utils.seed import set_seed

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
name = "change_detection_v2"
CROP = 256
VALSIZE = 512
BS = 16
EPOCHS = 80
PATIENCE = 20
TIME_BUDGET_H = 9.0

import torch as _torch
import torch.nn as _nn
import torch.nn.functional as _F
import segmentation_models_pytorch as _smp


class SiameseUNet(_nn.Module):
    """Shared-encoder siamese U-Net with fused pre/post features (fixed decoder)."""

    def __init__(self, encoder_name="resnet34", encoder_weights="imagenet",
                 fusion_mode="concat_absdiff", classes=1):
        super().__init__()
        self.fusion_mode = fusion_mode
        self.classes = classes

        base_unet = _smp.Unet(encoder_name=encoder_name, encoder_weights=encoder_weights,
                              in_channels=3, classes=classes)
        self.encoder = base_unet.encoder
        encoder_channels = self._get_encoder_channels()
        bottleneck_ch = encoder_channels[-1]

        if fusion_mode == "concat":
            fusion_mult = 2
        elif fusion_mode == "absdiff":
            fusion_mult = 1
        else:
            fusion_mult = 3   # concat_absdiff

        skip_channels = list(reversed(encoder_channels[:-1]))
        decoder_channels = [256, 128, 64, 32]

        self.skip_convs = _nn.ModuleList()
        for enc_ch in skip_channels:
            self.skip_convs.append(_nn.Sequential(
                _nn.Conv2d(enc_ch * fusion_mult, enc_ch, 1, bias=False),
                _nn.BatchNorm2d(enc_ch),
                _nn.ReLU(inplace=True),
            ))

        self.decoder = _nn.ModuleList()
        in_ch = bottleneck_ch * fusion_mult + skip_channels[0]
        for i, dec_ch in enumerate(decoder_channels):
            self.decoder.append(_nn.Sequential(
                _nn.Conv2d(in_ch, dec_ch, 3, padding=1, bias=False),
                _nn.BatchNorm2d(dec_ch),
                _nn.ReLU(inplace=True),
                _nn.Conv2d(dec_ch, dec_ch, 3, padding=1, bias=False),
                _nn.BatchNorm2d(dec_ch),
                _nn.ReLU(inplace=True),
            ))
            if i + 1 < len(decoder_channels):
                in_ch = dec_ch + skip_channels[i + 1]

        self.segmentation_head = _nn.Conv2d(decoder_channels[-1], classes, 1)

    def _get_encoder_channels(self):
        channels = []
        dummy = _torch.randn(1, 3, 64, 64)
        with _torch.no_grad():
            features = self.encoder(dummy)
        return [f.shape[1] for f in features]

    def _fuse(self, a, b):
        if self.fusion_mode == "concat":
            return _torch.cat([a, b], dim=1)
        if self.fusion_mode == "absdiff":
            return _torch.abs(b - a)
        return _torch.cat([a, b, _torch.abs(b - a)], dim=1)

    def forward(self, pre_image, post_image):
        features_pre = self.encoder(pre_image)
        features_post = self.encoder(post_image)
        x = self._fuse(features_pre[-1], features_post[-1])
        for i, (dec_block, skip_conv) in enumerate(zip(self.decoder, self.skip_convs)):
            x = _F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
            skip = skip_conv(self._fuse(features_pre[-(i + 2)], features_post[-(i + 2)]))
            if x.shape[2:] != skip.shape[2:]:
                x = _F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)
            x = dec_block(_torch.cat([x, skip], dim=1))
        if x.shape[2:] != pre_image.shape[2:]:
            x = _F.interpolate(x, size=pre_image.shape[2:], mode="bilinear", align_corners=False)
        return self.segmentation_head(x)


print("SiameseUNet (fixed decoder) defined")

class ChangeV2Dataset(Dataset):
    """Shift-corrected pre/post pairs; train = random crops (standard CD setting),
    val = fixed resize. Shifts come from phase correlation computed at preprocess time."""
    def __init__(self, root, split_csv, split="train", transform=None, apply_shift=True):
        self.root = Path(root)
        splits = pd.read_csv(split_csv)
        tiles = pd.read_csv(root / "metadata/tiles.csv").set_index("tile_id")
        rows = []
        for t in splits[splits["split"] == split]["tile_id"]:
            try:
                _find_tile(root / "images/pre", t)
            except FileNotFoundError:
                continue
            dy = dx = 0
            if apply_shift and t in tiles.index:
                _r = tiles.loc[t]
                _dy, _dx = _r.get("shift_dy"), _r.get("shift_dx")
                dy = 0 if _dy is None or pd.isna(_dy) else int(_dy)
                dx = 0 if _dx is None or pd.isna(_dx) else int(_dx)
            rows.append((t, dy, dx))
        self.rows = rows
        self.transform = transform
        print(f"ChangeV2Dataset split={split}: {len(rows)} tiles")

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        tid, dy, dx = self.rows[i]
        pre = _load_arr(_find_tile(self.root / "images/pre", tid))
        post = _load_arr(_find_tile(self.root / "images/post", tid))
        mask = _load_arr(self.root / "masks/change_binary" / f"{tid}.png").astype(np.uint8)
        if dy or dx:
            pre = np.roll(pre, (dy, dx), axis=(0, 1))
        pair = np.concatenate([pre, post], axis=2)
        if self.transform:
            aug = self.transform(image=pair, mask=mask)
            pair_t, m = aug["image"], aug["mask"]
        else:
            pair_t = torch.from_numpy(pair.transpose(2, 0, 1)); m = mask
        m = (m.long() if torch.is_tensor(m) else torch.from_numpy(np.asarray(m)).long()).float()
        return {"pre_image": pair_t[:3].float(), "post_image": pair_t[3:].float(),
                "mask": m, "tile_id": tid}

def change_transforms(phase="train"):
    mean, std = IMAGENET_MEAN * 2, IMAGENET_STD * 2
    if phase == "train":
        tf = [A.RandomCrop(CROP, CROP), A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
              A.RandomRotate90(p=0.5), A.RandomBrightnessContrast(p=0.3)]
    else:
        tf = [A.Resize(VALSIZE, VALSIZE)]
    tf += [A.Normalize(mean=mean, std=std), ToTensorV2()]
    return A.Compose(tf, is_check_shapes=False)

set_seed(42)
split_csv = PROCESSED / "metadata" / "splits_event_aware.csv"
train_ds = ChangeV2Dataset(PROCESSED, split_csv, "train", change_transforms("train"))
val_ds = ChangeV2Dataset(PROCESSED, split_csv, "val", change_transforms("val"))
test_ds = ChangeV2Dataset(PROCESSED, split_csv, "test", change_transforms("val"), apply_shift=True)
print(f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")
mark("chg2:data", extra=f"train={len(train_ds)}")

# pos_weight from the change masks (v1's unweighted BCE+Dice collapsed to all-zeros)
pos = tot = 0
for tid, _, _ in train_ds.rows[:300]:
    m = _load_arr(PROCESSED / "masks/change_binary" / f"{tid}.png")
    pos += int((m > 0).sum()); tot += m.size
_p = pos / max(tot, 1)
POS_W = float(np.clip((1 - _p) / max(_p, 1e-4), 2.0, 30.0))
print("change pixel fraction={:.4f} -> pos_weight={:.1f}".format(_p, POS_W))
mark("chg2:weights", extra="pos_frac={:.4f} pw={:.1f}".format(_p, POS_W))

class PosBCEDiceLoss(nn.Module):
    def __init__(self, pos_weight):
        super().__init__()
        self.register_buffer("pw", torch.tensor(float(pos_weight)))
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=self.pw)
        p = torch.sigmoid(logits)
        inter = (p * targets).sum(dim=(1, 2))
        card = p.sum(dim=(1, 2)) + targets.sum(dim=(1, 2))
        dice = (1 - (2 * inter + 1) / (card + 1)).mean()
        return 0.5 * bce + 0.5 * dice

loss_fn = PosBCEDiceLoss(POS_W).to(DEVICE)

train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True, num_workers=NW, pin_memory=False, persistent_workers=bool(NW))
val_loader = DataLoader(val_ds, batch_size=BS, shuffle=False, num_workers=NW, pin_memory=False, persistent_workers=bool(NW))
test_loader = DataLoader(test_ds, batch_size=BS, shuffle=False, num_workers=NW, pin_memory=False)

model = SiameseUNet(encoder_name="resnet34", encoder_weights="imagenet",
                    fusion_mode="concat_absdiff", classes=1).to(DEVICE)

try:
    _it = iter(train_loader)
    for _ in range(2):
        _b = next(_it)
        with torch.amp.autocast(DEVICE.type, enabled=(DEVICE.type == "cuda")):
            _lg = model(_b["pre_image"].to(DEVICE), _b["post_image"].to(DEVICE))
            _loss = loss_fn(_lg.squeeze(1), _b["mask"].to(DEVICE).float())
        _loss.backward()
        model.zero_grad(set_to_none=True)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
        mark("probe:ok", extra=f"peak={torch.cuda.max_memory_allocated()/1e9:.1f}GB")
    else:
        mark("probe:ok", extra="cpu")
except Exception:
    mark("probe:FAIL", extra=traceback.format_exc()[-1200:])
    raise

config = {"seed": 42, "device": DEVICE.type,
          "training": {"epochs": EPOCHS, "lr": 1e-4, "weight_decay": 1e-5, "scheduler": "cosine",
                       "mixed_precision": True, "early_stopping_patience": PATIENCE,
                       "save_best_metric": "iou", "time_budget_hours": TIME_BUDGET_H}}

mark("chg2:train-start", extra=f"train={len(train_ds)} epochs={EPOCHS} pw={POS_W:.1f}")
best = train_change_detection(model, train_loader, val_loader, loss_fn, config,
                              output_dir=str(Path("/tmp/run") / name))
print("best:", best)
mark("chg2:train-done", extra=str(best))

In [ ]:
import shutil as _shutil
_shutil.copytree(f"/tmp/run/change_detection_v2", WORK / "change_detection_v2", dirs_exist_ok=True)
print("results copied to", WORK / "change_detection_v2")
mark("copyout:change_detection_v2")
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

log_df = pd.read_csv(next((WORK / "change_detection_v2" / "logs").glob("*.csv")))
metric = "iou"
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(log_df["epoch"], log_df["train_loss"], label="train")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)
if metric in log_df.columns:
    axes[1].plot(log_df["epoch"], log_df[metric], color="tab:green")
    axes[1].set_title(f"val {metric} (best={log_df[metric].max():.4f} @ ep {int(log_df.loc[log_df[metric].idxmax(), 'epoch'])})")
axes[1].set_xlabel("epoch"); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(WORK / "change_detection_v2" / "training_curves.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
import json
# ---- shared eval kit: threshold sweep + approximate official xView2 score ----
import numpy as np
from scipy.ndimage import label as _cc_label

THRS = np.arange(0.20, 0.81, 0.05)

def sweep_loc_thresholds(probs, gts):
    """pooled inter/union per threshold -> {thr: IoU}"""
    out = {}
    p = np.concatenate(probs); g = np.concatenate(gts).astype(bool)
    for t in THRS:
        pred = p > t
        inter = np.logical_and(pred, g).sum()
        union = np.logical_or(pred, g).sum()
        out[round(float(t), 2)] = float(inter) / max(union, 1)
    return out

def best_thr(probs, gts):
    pooled = sweep_loc_thresholds(probs, gts)
    thr = max(pooled, key=pooled.get)
    return float(thr), pooled[thr], pooled

def _xview2_tile(loc_pred, loc_gt, dmg_pred, dmg_gt):
    """greedy component match: returns (fp, fn, matches=[(iou, pred_grade, gt_grade)])"""
    p_lab, np_ = _cc_label(loc_pred)
    g_lab, ng_ = _cc_label(loc_gt)
    fp = 0
    matches = []
    used = np.zeros(ng_ + 1, dtype=bool)
    if np_ and ng_:
        pair = p_lab.astype(np.int64) * (ng_ + 1) + g_lab
        hist = np.bincount(pair.ravel(), minlength=(np_ + 1) * (ng_ + 1)).reshape(np_ + 1, ng_ + 1)
        areas_p = hist[1:].sum(axis=1); areas_g = hist[:, 1:].sum(axis=0)
        for pi in range(np_):
            inters = hist[pi + 1, 1:]
            gi = int(np.argmax(inters))
            inter = int(inters[gi])
            if inter == 0:
                fp += 1
                continue
            union = int(areas_p[pi] + areas_g[gi] - inter)
            iou = inter / union
            if iou >= 0.5 and not used[gi]:
                used[gi] = True
                m = hist[pi + 1, gi + 1] > 0
                pg = dmg_pred[m]; gg = dmg_gt[m]
                pg = pg[pg > 0]; gg = gg[gg > 0]
                pgr = int(np.bincount(pg).argmax()) if len(pg) else 0
                ggr = int(np.bincount(gg).argmax()) if len(gg) else 0
                matches.append((iou, pgr, ggr))
            else:
                fp += 1
    else:
        fp += int(np_)
    fn = int((~used[1:]).sum())
    return fp, fn, matches

def xview2_score(tiles, thr):
    """tiles: [{loc_prob,loc_gt,dmg_pred,dmg_gt}] -> approximate official xView2 score."""
    stats = {t: [0, 0, 0] for t in (0.5, 0.6, 0.7, 0.8)}   # tp, fp, fn
    matches50 = []
    for d in tiles:
        pred = d["loc_prob"] > thr
        fp, fn, matches = _xview2_tile(pred, d["loc_gt"], d["dmg_pred"], d["dmg_gt"])
        for t in stats:
            tp_t = sum(1 for iou, _, _ in matches if iou >= t)
            fp_t = fp + sum(1 for iou, _, _ in matches if iou < t)
            fn_t = fn + sum(1 for iou, _, _ in matches if iou < t)
            stats[t][0] += tp_t; stats[t][1] += fp_t; stats[t][2] += fn_t
        matches50 += matches
    _f1 = lambda tp, fp, fn: 2 * tp / max(2 * tp + fp + fn, 1)
    loc_f1 = float(np.mean([_f1(*stats[t]) for t in stats]))
    per_class = {}
    for c in (1, 2, 3, 4):
        tp_c = sum(1 for _, p, g in matches50 if p == c and g == c)
        fp_c = sum(1 for _, p, g in matches50 if p == c and g != c)
        fn_c = sum(1 for _, p, g in matches50 if p != c and g == c)
        per_class[c] = round(2 * tp_c / max(2 * tp_c + fp_c + fn_c, 1), 4)
    dmg_f1 = float(np.mean(list(per_class.values())))
    return {"loc_f1": round(loc_f1, 4), "damage_f1": round(dmg_f1, 4),
            "overall_f1": round(0.3 * loc_f1 + 0.7 * dmg_f1, 4),
            "per_class_damage_f1": per_class,
            "n_pred_buildings": int(sum(s[0] + s[1] for s in stats.values()) // 8)}
print("eval kit ready")

ckpt = torch.load(WORK / name / "checkpoints" / "best.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.to(DEVICE).eval()
print("best ckpt: epoch", ckpt["epoch"], "iou", ckpt.get("best_iou"))

@torch.no_grad()
def change_probs(loader):
    model.eval()
    probs, gts = [], []
    for b in loader:
        with torch.amp.autocast(DEVICE.type, enabled=(DEVICE.type == "cuda")):
            out = model(b["pre_image"].to(DEVICE), b["post_image"].to(DEVICE))
        probs.append(torch.sigmoid(out.squeeze(1).float()).cpu().numpy())
        gts.append(b["mask"].numpy() > 0.5)
    return probs, gts

vp, vg = change_probs(val_loader)
val_sweep = sweep_loc_thresholds(vp, vg)
best_t = max(val_sweep, key=val_sweep.get)
thr = float(best_t); thr_iou = val_sweep[best_t]
print(f"val thr sweep: best={thr:.2f} IoU={thr_iou:.4f} | @0.5={val_sweep.get(0.5)}")

tp_, tg_ = change_probs(test_loader)
res = sweep_loc_thresholds(tp_, tg_)
print("TEST IoU @best thr:", res.get(best_t), "@0.5:", res.get(0.5))
pred = np.concatenate(tp_) > thr
gt = np.concatenate(tg_).astype(bool)
tpx = np.logical_and(pred, gt).sum(); fpx = np.logical_and(pred, ~gt).sum()
fnx = np.logical_and(~pred, gt).sum()
prec = tpx / max(tpx + fpx, 1); rec = tpx / max(tpx + fnx, 1)
f1c = 2 * tpx / max(2 * tpx + fpx + fnx, 1)
print(f"TEST @thr={thr}: precision={prec:.4f} recall={rec:.4f} F1={f1c:.4f} IoU={res.get(best_t):.4f}")

summary = {"run": name, "best_epoch": int(ckpt["epoch"]),
           "best_val_iou": float(ckpt.get("best_iou")),
           "pos_weight": POS_W,
           "threshold": thr,
           "val_iou_at_thr": round(float(thr_iou), 4),
           "test": {"iou": round(float(res.get(best_t, 0)), 4),
                    "iou_at_0.5": round(float(res.get(0.5, 0)), 4),
                    "precision": round(float(prec), 4), "recall": round(float(rec), 4),
                    "f1": round(float(f1c), 4)}}
with open(WORK / name / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

import numpy as np, matplotlib.pyplot as plt
batch = next(iter(test_loader))
with torch.no_grad():
    pr = (torch.sigmoid(model(batch["pre_image"].to(DEVICE), batch["post_image"].to(DEVICE))).squeeze(1) > thr).cpu().numpy()
def denorm(t):
    img = t.numpy().transpose(1, 2, 0)
    return (img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])).clip(0, 1)
fig, axes = plt.subplots(4, 4, figsize=(18, 18))
for i in range(min(4, batch["pre_image"].shape[0])):
    axes[0, i].imshow(denorm(batch["pre_image"][i])); axes[0, i].set_title(batch["tile_id"][i][:24])
    axes[1, i].imshow(denorm(batch["post_image"][i])); axes[1, i].set_title("post")
    axes[2, i].imshow(batch["mask"][i], cmap="gray"); axes[2, i].set_title("GT change")
    axes[3, i].imshow(pr[i], cmap="gray"); axes[3, i].set_title(f"pred @{thr}")
    for ax in axes[:, i]: ax.axis("off")
plt.tight_layout(); plt.savefig(WORK / name / "test_predictions.png", dpi=120, bbox_inches="tight")
plt.show()
mark("chg2:eval-done")